# Personal Finance Analyzer — Agent Loop

An agent that categorizes transactions, calculates totals,
identifies spending patterns, and proposes a savings plan — all autonomously via a tool loop.

In [ ]:
from rich.console import Console
from rich.table import Table
from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv(override=True)
openai = OpenAI()

In [ ]:
console = Console()

def show(text):
    try:
        console.print(text)
    except Exception:
        print(text)

## State

Simple Python lists that live outside the loop — the agent manipulates them through tools.

In [ ]:
expenses = []
incomes = []

## Tools

Four tools the agent can call:
1. **add_expense** — log an expense with category, amount, and description
2. **add_income** — log an income source and amount
3. **calculate_totals** — summarize everything and return a breakdown
4. **suggest_savings** — propose cuts to meet a monthly savings target

In [ ]:
def add_expense(category: str, amount: float, description: str) -> str:
    expenses.append({"category": category, "amount": amount, "description": description})
    console.print(f"[red]+ Expense:[/red] {description} — ₦{amount:.2f} [{category}]")
    return json.dumps({"status": "ok", "total_expenses": len(expenses)})


def add_income(source: str, amount: float) -> str:
    incomes.append({"source": source, "amount": amount})
    console.print(f"[green]+ Income:[/green] {source} — ₦{amount:.2f}")
    return json.dumps({"status": "ok", "total_incomes": len(incomes)})


def calculate_totals() -> str:
    total_income = sum(i["amount"] for i in incomes)
    total_spent = sum(e["amount"] for e in expenses)
    net = total_income - total_spent

    by_category = {}
    for e in expenses:
        by_category[e["category"]] = by_category.get(e["category"], 0) + e["amount"]

    table = Table(title="Financial Summary")
    table.add_column("Category", style="cyan")
    table.add_column("Amount", justify="right", style="red")
    table.add_column("% of Spending", justify="right")

    for cat, amt in sorted(by_category.items(), key=lambda x: -x[1]):
        pct = (amt / total_spent * 100) if total_spent > 0 else 0
        table.add_row(cat, f"₦{amt:.2f}", f"{pct:.1f}%")

    table.add_section()
    table.add_row("Total Spending", f"₦{total_spent:.2f}", "100%")
    table.add_row("Total Income", f"[green]₦{total_income:.2f}[/green]", "")
    table.add_row("Net", f"[{'green' if net >= 0 else 'red'}]₦{net:.2f}[/{'green' if net >= 0 else 'red'}]", "")
    console.print(table)

    result = {
        "total_income": total_income,
        "total_spent": total_spent,
        "net": net,
        "by_category": by_category
    }
    return json.dumps(result)


def suggest_savings(monthly_target: float) -> str:
    total_income = sum(i["amount"] for i in incomes)
    total_spent = sum(e["amount"] for e in expenses)
    current_savings = total_income - total_spent
    gap = monthly_target - current_savings

    by_category = {}
    for e in expenses:
        by_category[e["category"]] = by_category.get(e["category"], 0) + e["amount"]

    sorted_cats = sorted(by_category.items(), key=lambda x: -x[1])

    result = {
        "current_monthly_savings": current_savings,
        "target": monthly_target,
        "gap": gap,
        "spending_by_category_descending": sorted_cats,
        "on_track": gap <= 0
    }

    if gap <= 0:
        console.print(f"[bold green]Already saving ₦{current_savings:.2f}/mo — above the ₦{monthly_target:.2f} target![/bold green]")
    else:
        console.print(f"[bold yellow]Need to cut ₦{gap:.2f}/mo to hit ₦{monthly_target:.2f} target.[/bold yellow]")

    return json.dumps(result)

## Tool JSON schemas

These tell the LLM what tools are available and how to call them.

In [ ]:
add_expense_json = {
    "name": "add_expense",
    "description": "Log a single expense transaction with its category, dollar amount, and a short description",
    "parameters": {
        "type": "object",
        "properties": {
            "category": {
                "type": "string",
                "description": "Spending category (e.g. Housing, Food, Transport, Entertainment, Utilities, Subscriptions, Health, Shopping, Education, Other)"
            },
            "amount": {
                "type": "number",
                "description": "Dollar amount of the expense"
            },
            "description": {
                "type": "string",
                "description": "Brief description of the expense"
            }
        },
        "required": ["category", "amount", "description"],
        "additionalProperties": False
    }
}

add_income_json = {
    "name": "add_income",
    "description": "Log a single income entry with its source and dollar amount",
    "parameters": {
        "type": "object",
        "properties": {
            "source": {
                "type": "string",
                "description": "Source of income (e.g. Salary, Freelance, Investments, Side Hustle)"
            },
            "amount": {
                "type": "number",
                "description": "Dollar amount of the income"
            }
        },
        "required": ["source", "amount"],
        "additionalProperties": False
    }
}

calculate_totals_json = {
    "name": "calculate_totals",
    "description": "Calculate and return a full financial summary: total income, total spending, net savings, and spending broken down by category with percentages",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

suggest_savings_json = {
    "name": "suggest_savings",
    "description": "Given a monthly savings target, calculate the gap between current savings and the target, and return spending by category so the agent can recommend where to cut",
    "parameters": {
        "type": "object",
        "properties": {
            "monthly_target": {
                "type": "number",
                "description": "The desired monthly savings target in dollars"
            }
        },
        "required": ["monthly_target"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": add_expense_json},
    {"type": "function", "function": add_income_json},
    {"type": "function", "function": calculate_totals_json},
    {"type": "function", "function": suggest_savings_json}
]

## Tool dispatcher & agent loop

Uses `globals().get(tool_name)` for clean dispatch — no giant if/elif chain.

In [ ]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else json.dumps({"error": "unknown tool"})
        results.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
    return results


def loop(messages, model="gpt-4.1-mini"):
    done = False
    while not done:
        response = openai.chat.completions.create(model=model, messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

## System prompt

Tells the agent to work through a clear sequence: log income → log expenses → calculate totals → suggest savings → present a final analysis with actionable advice.

In [ ]:
system_message = """
You are a personal finance analyst agent. You are given a user's monthly financial data.
Your job is to:

1. Use add_income to log every income source the user mentions.
2. Use add_expense to log every expense, choosing an appropriate category for each.
3. Use calculate_totals to generate a full financial breakdown.
4. Use suggest_savings with a reasonable target (or the user's stated target) to see the gap.
5. After using all tools, provide a final analysis in Rich console markup (no code blocks) that includes:
   - A quick health check (are they spending more than they earn?)
   - Their top 3 spending categories and whether each seems reasonable
   - Specific, actionable recommendations to reduce spending (name exact items to cut or reduce)
   - A realistic monthly savings target based on their situation

Do not ask the user questions. Work with what you're given.
Be direct, specific, and encouraging — like a smart friend who's good with money.
"""

## Run it!

Here's a sample month of transactions. Swap these out with your own numbers to get a real analysis.

In [ ]:
expenses = []
incomes = []

user_message = """
Here's my finances for March 2026:

Income:
- Salary: $5,200
- Freelance web project: $800

Expenses:
- Rent: $1,800
- Groceries: $420
- Eating out / takeout: $380
- Electric bill: $95
- Internet: $60
- Phone plan: $45
- Netflix: $15.99
- Spotify: $10.99
- ChatGPT Plus: $20
- Gym membership: $50
- Gas: $160
- Car insurance: $140
- New sneakers: $130
- Concert tickets: $90
- Birthday gift for friend: $45
- Amazon random stuff: $210
- Uber rides (5 trips): $75
- Coffee shops: $65
- Doctor copay: $40
- Prescription: $25

I'd like to save $1,000/month. Is that possible? Where should I cut?
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message}
]

loop(messages)

## Try it with your own data

Replace the user message below with your actual monthly finances and run it again.

In [ ]:
expenses = []
incomes = []

your_message = """
Here's my finances for this month:

Income:
- PUT YOUR INCOME HERE

Expenses:
- PUT YOUR EXPENSES HERE

I'd like to save $500/month.
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": your_message}
]

# loop(messages)